# Training Data Generator - Cost Estimate

## Generating 5,000 Training Examples with GPT-4.1-mini

**Pricing (as of Nov 2024):**
- Input tokens: $0.40 per million tokens
- Output tokens: $1.60 per million tokens

**Estimated cost for 5,000 examples:**

Assuming each request uses:
- ~10,000 input tokens (example report + system prompt)
- ~5,000 output tokens (generated reasoning + verdict)

**Calculation:**
- Input: 5,000 × 10,000 = 50M tokens → 50 × $0.40 = **$20.00**
- Output: 5,000 × 5,000 = 25M tokens → 25 × $1.60 = **$40.00**

**Total Estimated Cost: $60.00** (approximately $40-60 USD depending on actual token usage)

---

**Rate Limiting & Retries:**
- Uses `limiter` package: 60 requests/minute with 120 burst capacity
- Automatic retries via `tenacity`: 3 attempts with exponential backoff (4-15 seconds)
- OpenAI client also configured with max_retries=3
- Estimated time: ~90 minutes for 5,000 examples (due to rate limiting)

In [1]:
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from typing import List, Literal
from enum import Enum
import json
from pathlib import Path
from tqdm.asyncio import tqdm_asyncio
import asyncio
from datetime import datetime
import os
import random
from limiter import Limiter
from tenacity import retry, stop_after_attempt, wait_exponential

client = AsyncOpenAI(
    timeout=60*5,
    max_retries=3,
)

# Rate limiter - adjust based on your OpenAI tier
limiter = Limiter(
    rate=2500,
    capacity=5000,  # burst capacity
)

In [2]:
class ProceedSignal(str, Enum):
    """Investment decision signal."""
    STRONG_YES = "strong_yes"
    QUESTIONABLE = "questionable"
    STRONG_NO = "strong_no"


class DealVerdict(BaseModel):
    """Structured verdict on a potential public equity investment."""
    
    proceed_signal: ProceedSignal = Field(
        description="Overall investment signal: strong_yes (high conviction opportunity), questionable (needs more analysis), or strong_no (pass on deal)"
    )
    
    evidence_pro: str = Field(
        description="Key positive factors supporting investment: competitive advantages, growth drivers, financial strengths, market opportunities",
    )
    
    evidence_contra: str = Field(
        description="Key concerns or risks against investment: competitive threats, financial weaknesses, market headwinds, execution risks",
    )
    
    similar_deals: str = Field(
        description="Comparable past investment deals (fictional examples) in similar sectors/situations that provide context. Format as brief descriptions of buyouts, growth equity, add-on acquisitions, etc."
    )
    
    key_information_requested: str = Field(
        description="Critical additional information, analysis, or due diligence needed to make final investment decision, even if signal is positive"
    )


class TrainingExample(BaseModel):
    """Complete training data example for fine-tuning."""
    
    report: str = Field(
        description="The generated company research report"
    )
    
    reasoning: str = Field(
        description="Conversational, first-person analytical reasoning in a thinking-aloud style. Use phrases like 'Okay, looking at this company...', 'Let me analyze the financials...', 'This company seems to...'. Walk through competition, customers, financials, and growth naturally."
    )
    
    verdict: DealVerdict = Field(
        description="The structured investment verdict with proceed signal and supporting evidence"
    )

In [ ]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=15),
    reraise=True,
)
async def generate_training_example(
    example_report_path: str,
    model: str = "gpt-4.1-mini",
    temperature: float = 1.0,
    client: AsyncOpenAI = None,
) -> TrainingExample:
    """
    Generate synthetic training data by creating a FAKE company report and corresponding verdict.
    
    Includes rate limiting via limiter and automatic retries via tenacity.
    Randomly selects a target verdict to ensure balanced distribution.
    """
    
    # Randomly select target verdict for balanced distribution
    target_verdict = random.choice(list(ProceedSignal))
    
    # Read the example report as a template
    example_report_text = Path(example_report_path).read_text()
    
    system_prompt = f"""You are a synthetic training data generator for equity investment research.

Your task is to generate COMPLETELY FICTIONAL training data consisting of:
1. A fictional company research report 
2. An investment verdict on that fake company
3. Analytical reasoning about that verdict

TARGET VERDICT: Your generated example should aim for a "{target_verdict.value}" verdict. Create a company and scenario that naturally leads to this conclusion.

The example report provided is ONLY a loose reference - DO NOT copy its structure rigidly. You must:

REPORT GENERATION:
- Invent a completely new company (different industry, different name, different business model)
- Generate realistic but fake financial data (revenue, margins, growth rates, balance sheet)
- Create fake competitors, customers, and market dynamics
- Make all facts, figures, and dates fictional but internally consistent
- Create a scenario that naturally leads to a "{target_verdict.value}" verdict

REPORT STRUCTURE FLEXIBILITY:
- The example structure and topics are JUST EXAMPLES - adapt freely to fit YOUR company
- Add company-specific sections when relevant (e.g., "Regulatory Environment" for healthcare, "Technology Stack" for SaaS, "Supply Chain" for manufacturing, "Unit Economics" for marketplaces)
- Remove sections that aren't relevant to your company
- Vary the depth and topics covered - sometimes focus more on Competition, other times on Customers or Growth
- GOAL IS DIVERSITY, NOT LENGTH - reports should be tailored to what's relevant for each company

Then, analyze YOUR OWN generated fake report through these lenses:
1. COMPETITION: Market position, competitive dynamics, defensibility
2. CUSTOMERS: Customer quality, retention, concentration risks  
3. FINANCIALS: Growth trajectory, profitability, cash generation, balance sheet
4. GROWTH OPPORTUNITIES: Organic growth, M&A, operational improvements

REASONING STYLE: Write your reasoning in a conversational, thinking-aloud first-person style. Example: "Okay, looking at this company... Let me analyze the financials here. This company seems to have strong margins, but..."

Finally, determine:
- proceed_signal: {target_verdict.value} (as targeted)
- evidence_pro: specific positive factors from YOUR fake report (3-6 points)
- evidence_contra: specific concerns from YOUR fake report (3-6 points)
- similar_deals: fictional investment deals that provide comps (e.g., "Thoma Bravo's $2.1B buyout of ConnectWise in 2019")
- key_information_requested: critical additional diligence needed

Generate diverse, realistic examples. Vary industries, company stages, financial profiles, report structures, and verdict outcomes."""

    user_prompt = f"""Here is an EXAMPLE report (DO NOT copy this structure or topics - adapt freely to fit your company):

{example_report_text}

---

Now generate a COMPLETELY NEW fictional company and deal memo with a structure and sections that fit YOUR company's industry and context. Then provide your reasoning and investment verdict for YOUR generated company."""
    
    async with limiter:
        response = await client.responses.parse(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            text_format=TrainingExample,
        )
    
    result = response.output_parsed
    
    # Save individual example to file

    output_dir = "../data/output/individual"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S-%f")[:-3]  # milliseconds
    output_file = os.path.join(output_dir, f"{timestamp}.json")
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result.model_dump(), f, indent=2)
    
    return result

In [4]:
async def generate_batch(
    example_report_path: str,
    num_examples: int = 10,
    client: AsyncOpenAI = None,
    temperature: float = 1.5,
) -> List[TrainingExample]:
    """
    Generate multiple DIVERSE training examples using the same report as a template.
    
    Each example will be a completely different fake company with unique:
    - Company name and industry
    - Financial data
    - Competitive landscape
    - Investment verdict
    
    Args:
        example_report_path: Path to example report (used as structure template only)
        num_examples: Number of diverse fake companies to generate
        client: AsyncOpenAI client instance
        temperature: Sampling temperature (high = more diversity)
        
    Returns:
        List of TrainingExample objects, each with a unique fake company
    """
    import asyncio
    
    tasks = [
        generate_training_example(
            example_report_path=example_report_path,
            temperature=temperature,
            client=client
        )
        for _ in range(num_examples)
    ]
    
    results = await tqdm_asyncio.gather(*tasks, desc="Generating training examples")
    return results

In [5]:
def save_training_data(
    examples: List[TrainingExample],
    output_path: str = "training_data.jsonl"
):
    """
    Save training examples to JSONL format for fine-tuning.
    
    Args:
        examples: List of TrainingExample objects
        output_path: Path to output JSONL file
    """
    with open(output_path, 'w', encoding="utf-8") as f:
        for example in examples:
            # Convert to dict and write as JSON line
            f.write(example.model_dump_json() + '\n')
    
    print(f"Saved {len(examples)} training examples to {output_path}")

In [21]:
batch = await generate_batch(
    example_report_path="../data/output/Duolingo_2025-11-18_21-47-42.md",
    num_examples=1000,
    client=client,
    temperature=1.0
)

Generating training examples: 100%|██████████| 1000/1000 [00:53<00:00, 18.66it/s]


Save all individual ones to training JSON

In [22]:
# Load and validate all existing training examples from individual files
individual_dir = Path("../data/output/individual")
all_examples = []

# Read all JSON files from the individual directory
for json_file in individual_dir.glob("*.json"):
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # Validate and convert to TrainingExample
            example = TrainingExample(**data)
            all_examples.append(example)
    except Exception as e:
        print(f"Error loading {json_file.name}: {e}")

print(f"Loaded {len(all_examples)} existing training examples from individual files")

Loaded 5154 existing training examples from individual files


In [23]:
# Save to file
save_training_data(all_examples, "../data/training_data_examples_all.jsonl")

Saved 5154 training examples to ../data/training_data_examples_all.jsonl


Stats

In [32]:
from collections import Counter

# Count verdict statistics
verdict_counts = Counter([example.verdict.proceed_signal.value for example in all_examples])
print("Verdict Distribution:")
for verdict, count in verdict_counts.most_common():
    print(f"  {verdict}: {count} ({count/len(all_examples)*100:.1f}%)")

Verdict Distribution:
  questionable: 1768 (34.3%)
  strong_yes: 1756 (34.1%)
  strong_no: 1630 (31.6%)


Explore an example

In [25]:
print(all_examples[111].report[:1000])

Velodigm Therapeutics – Clinical-Stage Biotech Research Memo
Date: April 15, 2026

I. Company Overview and Business Model

– Velodigm Therapeutics is a U.S.-based biotechnology startup focused on developing RNA interference (RNAi) therapeutics targeting neurodegenerative diseases, primarily Alzheimer’s and Parkinson’s. Founded in 2016, headquartered in Cambridge, MA.
– Privately held; currently in Series D financing round seeking $125M to fund Phase 2 clinical trials.
– Development pipeline:
  • VDT-101: RNAi candidate targeting Tau protein in Alzheimer’s disease; Phase 1b completed with safety signals observed but no clear efficacy.
  • VDT-202: Candidate aimed at alpha-synuclein suppression in Parkinson’s; preclinical stage with IND planned late 2026.
  • Platform technology: Proprietary lipid nanoparticle delivery system aimed to improve brain targeting; early development with significant technical challenges.

– Business model: R&D-centric with no commercial products; relying on mi

In [26]:
print(all_examples[111].reasoning[:1000])

Okay, looking at Velodigm Therapeutics, this is a clinical-stage biotech with a focus on RNAi therapies targeting neurodegenerative diseases such as Alzheimer’s and Parkinson’s. Let me analyze the competitive and scientific landscape first.

The market for these indications is indeed huge, but RNAi delivery into the central nervous system remains notoriously difficult. Velodigm’s platform to do this with lipid nanoparticles is innovative but currently unproven in humans. The Phase 1b data on VDT-101 showed some safety concerns and no clear efficacy, which is concerning given the high bar for neurodegeneration drugs.

Competition is intense, with established big pharmas like Biogen having more advanced monoclonal antibodies and ASOs in later stages. Also, other RNAi or gene therapy players have better clinical progress. This makes Velodigm’s chances to differentiate and capture market share quite slim.

From a customer perspective, there is no revenue yet—patients are not direct custome

In [27]:
all_examples[111].verdict.proceed_signal.value

'strong_no'

In [28]:
all_examples[111].verdict.evidence_pro

'1. Large potential markets: Alzheimer’s ($15B) and Parkinson’s ($7B) represent significant unmet needs.\n2. Proprietary lipid nanoparticle delivery platform aimed at overcoming CNS delivery challenge.\n3. Experienced scientific leadership with some track record in biotech innovation.\n4. Intellectual property portfolio with 15 patents covering RNAi sequences and delivery.\n5. Early clinical progress includes completion of Phase 1b for lead candidate VDT-101.'

In [29]:
all_examples[111].verdict.evidence_contra

'1. Lack of clinical efficacy signal and presence of safety concerns in Phase 1b for VDT-101.\n2. Immature and unproven CNS delivery platform; high technical risk.\n3. Intense competition from established pharma with more advanced therapies (e.g., Biogen, Roche).\n4. High cash burn with limited runway; reliance on large Series D raise amidst clinical and scientific uncertainty.\n5. Preclinical status of second candidate VDT-202 delays pipeline diversification.\n6. Questionable commercial viability given crowded neurodegeneration market and uncertain differentiation.'

In [30]:
all_examples[111].verdict.similar_deals

"– Silverline Biotherapeutics $130M Series C in 2023 for ASO therapies targeting Huntington’s disease but later delayed clinical starts due to safety concerns.\n– Helixion Pharma’s $100M Series B in 2022 focusing on mRNA modalities for rare diseases, with similar CNS delivery challenges.\n– Nexora Therapeutics $80M growth equity in 2024 targeting oligonucleotide platforms but with mixed preclinical data and long timelines.\n– Notixis Capital's early-stage buyout of NeuroGene in 2021, which failed due to poor Phase 2 data and was written down heavily.\n– Altair Bioscience $50M Series D in 2025 focusing on gene editing for Parkinson’s, facing regulatory and delivery hurdles analogous to Velodigm."

In [31]:
all_examples[111].verdict.key_information_requested

'– Detailed Phase 1b clinical data including safety, biomarker, and exploratory efficacy results.\n– Independent CNS exposure and target engagement data for VDT-101 and delivery platform.\n– Comprehensive competitive landscape analysis comparing Velodigm’s platform to alternatives.\n– Financial projections and milestone-based cash flow models for current trials and fundraising assumptions.\n– Partner interest indications and licensing terms explored.\n– Intellectual property freedom-to-operate analysis especially regarding overlapping nanoparticle delivery patents.\n– Risk mitigation plans and contingency strategies if current platform technology does not meet clinical milestones.'

Read training data JSONL

In [35]:
# Read all training examples from JSONL file
with open('../data/training_data_examples_all.jsonl', 'r') as f:
    training_data = [json.loads(line) for line in f]

print(f"Loaded {len(training_data)} training examples")

Loaded 5154 training examples


Count tokens - approximate our training / testing size

In [38]:
import tiktoken
encoding = tiktoken.get_encoding("o200k_base")

In [46]:
all_tokens = len(encoding.encode(str(training_data)))
print(f"Total tokens: {all_tokens}")

TRAIN_SIZE = 0.7
EVAL_SIZE = 0.15
TEST_SIZE = 1-TRAIN_SIZE-EVAL_SIZE

# Calculate training / testing split
training_tokens = int(all_tokens * TRAIN_SIZE)
eval_tokens = int(all_tokens * EVAL_SIZE)
test_tokens = all_tokens - training_tokens - eval_tokens
print(f"Training tokens: {training_tokens}")
print(f"Eval tokens: {eval_tokens}")
print(f"Test tokens: {test_tokens}")

Total tokens: 10049097
Training tokens: 7034367
Eval tokens: 1507364
Test tokens: 1507366
